# Milestone M3 - Eksperimen Utama (3 Seed), 4 Ablasi Anggota, Error Gallery & Grad-CAM (Topik T3-A)
Notebook ini menjalankan seluruh eksperimen terkontrol untuk Milestone M3 secara lengkap dan reproduksibel:
1. **Perbandingan Utama Multi-Seed (`3 Seed: 42, 123, 999` × `2 Paradigma` × `2 Arah` × `4 Strategi Warna` = 48 Run):** `U-Net` vs `DeepLabV3+` pada `D1 -> D2` dan `D2 -> D1` untuk membandingkan `NoAdapt`, `ColorJitter`, `Macenko` (white-balanced), dan `Reinhard` (`mean ± std` K4/K5).
2. **4 Studi Ablasi Satu-Faktor (1 Per Anggota Kelompok 6, `seed=42`):**
   - *Attala (`230441100144`):* Strategi adaptasi warna + sensitivitas normalisasi kanonik (`train+eval` vs `eval-only`)
   - *Nafaul (`240441100125`):* Fungsi loss (`CE` vs `Dice` vs `CE+Dice`)
   - *M. Rafly (`240441100086`):* Resolusi masukan & augmentasi geometrik (`128x128` vs `256x256`, `none` vs `geom`)
   - *Naufal (`240441100038`):* Encoder & strategi fine-tuning (`ResNet-34 Full` vs `MobileNetV2 Full` vs `ResNet-34 Frozen`)
3. **Analisis Kesalahan Kuantitatif (`Baseline` vs `Adaptasi Terbaik`), Galeri Kesalahan (min. 8 kasus), & Grad-CAM (`Nukleus` dan `Sitoplasma`, 4 benar + 4 salah)** + **Tabel Efisiensi (`K7`)**.


In [ ]:
# Clone / pull repositori di Google Colab & pasang dependensi
import os, sys, subprocess

REPO_URL = "https://github.com/attaramadhani/UAS-DL-Kelompok6-T3A-WBC-DomainShift.git"
REPO_NAME = "UAS-DL-Kelompok6-T3A-WBC-DomainShift"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_NAME) and os.path.basename(os.getcwd()) != REPO_NAME:
        subprocess.run(["git", "clone", REPO_URL], check=True)
    if os.path.exists(REPO_NAME) and os.path.basename(os.getcwd()) != REPO_NAME:
        os.chdir(REPO_NAME)
    subprocess.run(["git", "pull", "--ff-only"], check=False)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Persiapan Dataset, Split Tetap (`seed=42`) & Referensi Normalisasi dari Set Train


In [ ]:
import csv, gc, os, time, zipfile
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
from torch.utils.data import DataLoader

from src.dataset import download_and_extract_zheng, generate_fixed_splits, make_torch_dataset
from src.stain_norm import MacenkoNormalizer, ReinhardNormalizer, select_representative_reference
from src.models import build_segmentation_model
from src.trainer import evaluate_loader, run_experiment
from src.xai_and_errors import (
    generate_gradcam_grid,
    get_gradcam_target_layer,
    save_error_gallery,
    summarize_error_categories,
)

domain_dirs = download_and_extract_zheng("data")
split_csvs = {"D1": "splits/zheng_d1_split.csv", "D2": "splits/zheng_d2_split.csv"}
if not all(os.path.exists(p) for p in split_csvs.values()):
    split_csvs = generate_fixed_splits(domain_dirs, "splits", seed=42)
print("Menggunakan split tetap:", split_csvs)

def get_train_paths(dkey):
    with open(split_csvs[dkey], encoding="utf-8") as f:
        return [os.path.join(domain_dirs[dkey], r["image_file"]) for r in csv.DictReader(f) if r["split"] == "train"]

# Fit normalizer HANYA dari satu citra representatif set train domain sumber (nol kebocoran data uji)
ref_train = {dkey: select_representative_reference(get_train_paths(dkey)) for dkey in ("D1", "D2")}
norm_macenko = {dkey: MacenkoNormalizer().fit(ref_train[dkey]) for dkey in ("D1", "D2")}
norm_reinhard = {dkey: ReinhardNormalizer().fit(ref_train[dkey]) for dkey in ("D1", "D2")}

LOG_PATH = "logs/m3_experiment_runs.csv"
RESUME_EXISTING = True  # Set False jika ingin mengulang seluruh log dari nol

completed_runs = set()
if RESUME_EXISTING and os.path.exists(LOG_PATH):
    prev_df = pd.read_csv(LOG_PATH)
    prev_ok = prev_df[prev_df["status"] == "success"]
    prev_ok.to_csv(LOG_PATH, index=False)
    completed_runs = set(prev_ok["run_name"].tolist())
    print(f"Resume aktif: {len(completed_runs)} run sukses sebelumnya dipertahankan.")
elif os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)

print("Siap menjalankan eksperimen M3 pada split tetap seed=42!")


## 2. Perbandingan Utama (2 Paradigma × 4 Strategi Warna × 2 Arah × 3 Seed = 48 Run)


In [ ]:
SEEDS = [42, 123, 999]
EPOCHS = 40
t_suite_start = time.perf_counter()

COLOR_STRATEGIES = [
    ("NoAdapt", "geom", None),
    ("ColorJitter", "color_geom", None),
    ("Macenko", "geom", "macenko"),
    ("Reinhard", "geom", "reinhard"),
]

main_plan = []
for train_dom, target_dom in [("D1", "D2"), ("D2", "D1")]:
    for arch in ["unet", "deeplabv3plus"]:
        for strat_name, aug_mode, norm_kind in COLOR_STRATEGIES:
            for seed in SEEDS:
                main_plan.append((train_dom, target_dom, arch, strat_name, aug_mode, norm_kind, seed))

for idx, (train_dom, target_dom, arch, strat_name, aug_mode, norm_kind, seed) in enumerate(main_plan, 1):
    name = f"Main_{arch}_{strat_name}_{train_dom}_to_{target_dom}_s{seed}"
    if name in completed_runs:
        print(f"[{idx}/{len(main_plan)}] SKIP (sudah selesai): {name}")
        continue

    norm_obj = None
    if norm_kind == "macenko":
        norm_obj = norm_macenko[train_dom]
    elif norm_kind == "reinhard":
        norm_obj = norm_reinhard[train_dom]

    res = run_experiment(
        run_name=name,
        owner="Tim & Attala (230441100144)",
        strategy=strat_name,
        domain_dirs=domain_dirs,
        split_csvs=split_csvs,
        train_domain=train_dom,
        target_domain=target_dom,
        arch=arch,
        encoder_name="resnet34",
        loss_name="ce_dice",
        aug_mode=aug_mode,
        train_normalizer=norm_obj,
        eval_normalizer=norm_obj,
        seed=seed,
        epochs=EPOCHS,
        log_csv_path=LOG_PATH,
        save_weights_path=f"weights/{name}.pth" if seed == 42 else None,
        capture_test_records=False,
    )
    row = res["row"]
    elapsed_min = (time.perf_counter() - t_suite_start) / 60.0
    print(
        f"[{idx}/{len(main_plan)}] {name} | val_dice={row['val_dice_fg']:.4f} | "
        f"in_miou={row['in_miou_fg']:.4f} | cross_miou={row['cross_miou_fg']:.4f} "
        f"({row['duration_sec']:.1f}s, total {elapsed_min:.1f}m)"
    )
    del res
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_runs = pd.read_csv(LOG_PATH)
metric_cols = [
    "in_dice_nuc", "in_dice_cyto", "in_iou_nuc", "in_iou_cyto", "in_miou_fg", "in_hd95_nuc", "in_hd95_cyto",
    "cross_dice_nuc", "cross_dice_cyto", "cross_iou_nuc", "cross_iou_cyto", "cross_miou_fg", "cross_hd95_nuc", "cross_hd95_cyto",
]
summary_main = (
    df_runs[df_runs["run_name"].str.startswith("Main_")]
    .groupby(["train_domain", "target_domain", "arch", "strategy"])[metric_cols]
    .agg(["mean", "std"])
)
os.makedirs("logs", exist_ok=True)
summary_main.to_csv("logs/m3_main_summary.csv")
display(summary_main)


## 3. Studi Ablasi Satu-Faktor Masing-Masing Anggota Kelompok 6 (`seed=42`) & Tabel Efisiensi (K7)


In [ ]:
ablation_queue = [
    # --- ABLASI 1 TAMBAHAN (Attala): Sensitivitas normalisasi Eval-Only vs Kanonik (Train+Eval) ---
    dict(
        run_name="Abl_Attala_Macenko_EvalOnly",
        owner="Attala (230441100144)",
        strategy="Macenko_EvalOnly",
        train_domain="D1", target_domain="D2", arch="unet", encoder_name="resnet34",
        freeze_encoder=False, loss_name="ce_dice", aug_mode="geom",
        train_normalizer=None, eval_normalizer=norm_macenko["D1"], input_size=256,
    ),
    dict(
        run_name="Abl_Attala_Reinhard_EvalOnly",
        owner="Attala (230441100144)",
        strategy="Reinhard_EvalOnly",
        train_domain="D1", target_domain="D2", arch="unet", encoder_name="resnet34",
        freeze_encoder=False, loss_name="ce_dice", aug_mode="geom",
        train_normalizer=None, eval_normalizer=norm_reinhard["D1"], input_size=256,
    ),
    # --- ABLASI 2 (Nafaul): Fungsi Loss pada U-Net D1 -> D2 ---
    *[
        dict(
            run_name=f"Abl_Nafaul_Loss_{lname}",
            owner="Nafaul (240441100125)",
            strategy=f"Loss_{lname}",
            train_domain="D1", target_domain="D2", arch="unet", encoder_name="resnet34",
            freeze_encoder=False, loss_name=lname, aug_mode="geom",
            train_normalizer=None, eval_normalizer=None, input_size=256,
        )
        for lname in ["ce", "dice", "ce_dice"]
    ],
    # --- ABLASI 3 (M. Rafly): Resolusi Masukan & Augmentasi Geometrik pada DeepLabV3+ ---
    *[
        dict(
            run_name=f"Abl_Rafly_Sz{sz}_{amode}",
            owner="M. Rafly (240441100086)",
            strategy=f"Sz{sz}_{amode}",
            train_domain="D1", target_domain="D2", arch="deeplabv3plus", encoder_name="resnet34",
            freeze_encoder=False, loss_name="ce_dice", aug_mode=amode,
            train_normalizer=None, eval_normalizer=None, input_size=sz,
        )
        for sz, amode in [(128, "geom"), (256, "none"), (256, "geom")]
    ],
    # --- ABLASI 4 (Naufal): Encoder & Strategi Fine-Tuning pada DeepLabV3+ ---
    *[
        dict(
            run_name=f"Abl_Naufal_{enc}_{'Frozen' if frz else 'Full'}",
            owner="Naufal (240441100038)",
            strategy=f"{enc}_{'Frozen' if frz else 'Full'}",
            train_domain="D1", target_domain="D2", arch="deeplabv3plus", encoder_name=enc,
            freeze_encoder=frz, loss_name="ce_dice", aug_mode="geom",
            train_normalizer=None, eval_normalizer=None, input_size=256,
        )
        for enc, frz in [("resnet34", False), ("mobilenet_v2", False), ("resnet34", True)]
    ],
]

for idx, cfg in enumerate(ablation_queue, 1):
    if cfg["run_name"] in completed_runs:
        print(f"[{idx}/{len(ablation_queue)}] SKIP (sudah selesai): {cfg['run_name']}")
        continue
    res = run_experiment(
        domain_dirs=domain_dirs,
        split_csvs=split_csvs,
        seed=42,
        epochs=EPOCHS,
        log_csv_path=LOG_PATH,
        save_weights_path=f"weights/{cfg['run_name']}.pth",
        capture_test_records=False,
        **cfg,
    )
    row = res["row"]
    print(
        f"[{idx}/{len(ablation_queue)}] {cfg['run_name']} | val_dice={row['val_dice_fg']:.4f} | "
        f"in_miou={row['in_miou_fg']:.4f} | cross_miou={row['cross_miou_fg']:.4f} ({row['duration_sec']:.1f}s)"
    )
    del res
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_all = pd.read_csv(LOG_PATH)
print(f"Total waktu GPU seluruh eksperimen: {df_all['duration_sec'].sum() / 60.0:.2f} menit")

# Tabel Efisiensi Model (K7: batch=1 setelah 10 warm-up pada 50 citra)
eff_cols = ["run_name", "arch", "encoder", "freeze_encoder", "input_size", "params_m", "latency_ms", "gpu"]
display(df_all[df_all["seed"] == 42][eff_cols].drop_duplicates(subset=["arch", "encoder", "freeze_encoder", "input_size"]))


## 4. Analisis Kesalahan Kuantitatif (`Baseline` vs `Adaptasi Terbaik`), Galeri Kesalahan & Grad-CAM (`Nukleus` dan `Sitoplasma`)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
df_all = pd.read_csv(LOG_PATH)

# Evaluasi ulang checkpoint beku seed=42 untuk inspeksi XAI & error gallery (post-hoc tanpa tuning)
def evaluate_saved_run(run_row, dkey="D2"):
    norm_map = {
        "Macenko": norm_macenko["D1"],
        "Reinhard": norm_reinhard["D1"],
        "Macenko_EvalOnly": norm_macenko["D1"],
        "Reinhard_EvalOnly": norm_reinhard["D1"],
    }
    ev_norm = norm_map.get(run_row["strategy"], None)
    test_ds = make_torch_dataset(
        domain_dirs[dkey], split_csvs[dkey], "test",
        input_size=int(run_row["input_size"]), normalizer=ev_norm,
    )
    loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
    model = build_segmentation_model(
        arch=run_row["arch"],
        encoder_name=run_row["encoder"],
        encoder_weights=None,
        classes=3,
        freeze_encoder=bool(run_row["freeze_encoder"]),
    ).to(device)
    ckpt = f"weights/{run_row['run_name']}.pth"
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    summary, records = evaluate_loader(model, loader, device=device, capture_records=True, compute_hd95=True)
    for r in records:
        r["domain"] = f"{dkey} Cross-Domain ({run_row['strategy']})"
    return model, summary, records

# 1. Baseline (NoAdapt) D1 -> D2 (U-Net seed=42)
base_row = df_all[df_all["run_name"] == "Main_unet_NoAdapt_D1_to_D2_s42"].iloc[0]
base_model, _, base_records = evaluate_saved_run(base_row, "D2")

# 2. Pilih secara otomatis strategi warna terbaik pada U-Net D1 -> D2 (seed=42) berdasar hasil evaluasi beku
cand_rows = df_all[
    (df_all["run_name"].str.startswith("Main_unet_"))
    & (df_all["train_domain"] == "D1")
    & (df_all["target_domain"] == "D2")
    & (df_all["seed"] == 42)
]
best_row = cand_rows.sort_values("cross_miou_fg", ascending=False).iloc[0]
print(f"Model U-Net D1->D2 (seed=42) terbaik terpilih otomatis untuk analisis XAI: {best_row['run_name']} (cross_miou_fg={best_row['cross_miou_fg']:.4f})")
best_model, _, best_records = evaluate_saved_run(best_row, "D2")

os.makedirs("docs/figures", exist_ok=True)
print("\n--- Frekuensi Kategori Kesalahan di Seluruh 15 Citra D2 Test ---")
print("Baseline (NoAdapt):", summarize_error_categories(base_records))
print(f"Adaptasi Terbaik ({best_row['strategy']}):", summarize_error_categories(best_records))

# Galeri kesalahan min. 8 kasus untuk Baseline dan Best Adapted
save_error_gallery(base_records, "docs/figures/fig4a_error_gallery_baseline.png", top_k=8)
save_error_gallery(best_records, "docs/figures/fig4b_error_gallery_best_adapted.png", top_k=8)
print("=== Galeri Kesalahan Baseline (NoAdapt) ===")
display(Image(filename="docs/figures/fig4a_error_gallery_baseline.png"))
print(f"=== Galeri Kesalahan Adaptasi Terbaik ({best_row['strategy']}) ===")
display(Image(filename="docs/figures/fig4b_error_gallery_best_adapted.png"))

# Grad-CAM: 4 kasus benar/terbaik + 4 kasus salah/terburuk untuk Nukleus (2) & Sitoplasma (1)
sorted_rec = sorted(best_records, key=lambda r: r["dice_fg_mean"])
worst_4 = [{**r, "tag": "Salah/Terburuk"} for r in sorted_rec[:4]]
best_4 = [{**r, "tag": "Benar/Terbaik"} for r in sorted_rec[-4:]]
target_layer = get_gradcam_target_layer(best_model)

generate_gradcam_grid(best_model, target_layer, best_4 + worst_4, "docs/figures/fig5a_gradcam_nucleus.png", target_class=2, device=device)
generate_gradcam_grid(best_model, target_layer, best_4 + worst_4, "docs/figures/fig5b_gradcam_cytoplasm.png", target_class=1, device=device)
print("=== Grad-CAM Nukleus (Kelas 2) ===")
display(Image(filename="docs/figures/fig5a_gradcam_nucleus.png"))
print("=== Grad-CAM Sitoplasma (Kelas 1) ===")
display(Image(filename="docs/figures/fig5b_gradcam_cytoplasm.png"))

base_model.cpu(); best_model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. Grafik Ringkasan Domain Shift (3-Seed `mean ± std`) & Arsip Unduhan Output


In [ ]:
df_main = pd.read_csv(LOG_PATH)
df_main = df_main[df_main["run_name"].str.startswith("Main_")].copy()
df_main["direction"] = df_main["train_domain"] + " -> " + df_main["target_domain"]

strategies_order = ["NoAdapt", "ColorJitter", "Macenko", "Reinhard"]
colors = {"NoAdapt": "#94A3B8", "ColorJitter": "#F59E0B", "Macenko": "#2563EB", "Reinhard": "#10B981"}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
for ax, direction in zip(axes, ["D1 -> D2", "D2 -> D1"]):
    sub = df_main[df_main["direction"] == direction]
    x = np.arange(2)
    width = 0.19
    for s_idx, strat in enumerate(strategies_order):
        means, stds = [], []
        for arch in ["unet", "deeplabv3plus"]:
            vals = sub[(sub["arch"] == arch) & (sub["strategy"] == strat)]["cross_miou_fg"]
            means.append(vals.mean())
            stds.append(vals.std(ddof=1) if len(vals) > 1 else 0.0)
        offset = (s_idx - 1.5) * width
        ax.bar(x + offset, means, width, yerr=stds, capsize=3, label=strat, color=colors[strat])
    ax.set_xticks(x)
    ax.set_xticklabels(["U-Net (ResNet-34)", "DeepLabV3+ (ResNet-34)"])
    ax.set_title(f"Cross-Domain mIoU Foreground ({direction}, 3 Seed mean±std)")
    ax.set_ylabel("mIoU Foreground (0-1)")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("docs/figures/fig6_domain_shift_comparison.png", dpi=200, bbox_inches="tight")
plt.close(fig)
display(Image(filename="docs/figures/fig6_domain_shift_comparison.png"))

# Kemas seluruh hasil log & grafik agar mudah diunduh dari Colab atau disinkronkan ke GitHub
bundle_path = "m3_results_bundle.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("logs", "docs/figures"):
        for root, _, files in os.walk(folder):
            for fname in files:
                fpath = os.path.join(root, fname)
                zf.write(fpath)
print("Selesai! Arsip hasil M3 siap diunduh di:", os.path.abspath(bundle_path))
